In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data", "TRACER")
dataType = "RadarData"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

spinup_hours = 24
# spinup_hours = 12

# RunType = ("TRACER","MOIST","NSSL",spinup_hours)
RunType = ("TRACER","DRY","NSSL",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
########################
#DATA INFORMATION

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. X-Band Scanning ARM Cloud Radar (XSACRCFRQC), 2022-06-09 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by Y. Feng, A. Matthews, E. Schuman, K. Johnson, I. Lindenmaier, V. Castro and T. Wendler. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/2001296.

#Globus Download Link
# https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261781*__;Ly8v!!PvDODwlR4mBZyAb0!REKAOzHJNvJk50CY5Pjl135CV83BhArwtdyMDuBM-28KqreBug8Xb5Mc3MLgy_p9PUiWe2uVXq-EfUtLcGdH5w$

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. Ka-Band Scanning ARM Cloud Radar (KASACRCFRQC), 2022-06-08 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by I. Lindenmaier, K. Johnson, D. Nelson, A. Matthews, T. Wendler, V. Melo de Castro, M. Rocque and Y. Feng. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/1877338.

# https://armgov.svcs.arm.gov/capabilities/instruments/kasacr

#Globus Download Link
#https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261893*__;Ly8v!!PvDODwlR4mBZyAb0!UJrKWg_xaYuaaa_iaY91TCVMB_neSskXsDKHtPPCG7Ix6sEmiEvnngTUrYuV18LudZqxB3eHyTDuCHSZ3o3zrg$

In [ ]:
#LOADING RADAR CLASS

fileDirectory = "/glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA/Observation_Data/TRACER/ARM_XBandRadar_Data/ppiv/"
radarDataType = "X-Band"
RadarData_XBand = RadarData_ARM_Class(fileDirectory,radarDataType)

fileDirectory = "/glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA/Observation_Data/TRACER/ARM_KBandRadar_Data/ppiv/"
radarDataType = "K-Band"
RadarData_KBand = RadarData_ARM_Class(fileDirectory,radarDataType)

In [ ]:
##########################
#ORIGINAL RADAR DATA

In [ ]:
#GETTING DATA
target_time = datetime(2022, 7, 1, 0, 0, 0)

#TRACER X-Band Radar
index, _ = RadarData_XBand.FindClosestFileIndex(RadarData_XBand.fileList, target_time)
filePath_XBand = RadarData_XBand.filePathList[index]
radar_XBand = RadarData_XBand.GetRadarData(filePath_XBand)

#TRACER K-Band Radar
index, _ = RadarData_KBand.FindClosestFileIndex(RadarData_KBand.fileList, target_time)
filePath_KBand = RadarData_KBand.filePathList[index]
radar_KBand = RadarData_KBand.GetRadarData(filePath_KBand)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
RadarData_XBand.PlotReflectivity(axis=axes[0])
RadarData_KBand.PlotReflectivity(axis=axes[1])

In [ ]:
##########################
#GRIDDED RADAR DATA

In [ ]:
#LOADING DATA
grid_XBand, data_XBand = RadarData_XBand.GridRadarReflectivity(radar_XBand)
grid_KBand, data_KBand = RadarData_XBand.GridRadarReflectivity(radar_KBand)

In [ ]:
#PLOTTING DATA

fig, axes = plt.subplots(1, 2, figsize=(12,4))
RadarData_XBand.PlotGriddedReflectivity(grid_XBand,data_XBand, axis=axes[0])
RadarData_KBand.PlotGriddedReflectivity(grid_KBand,data_KBand, axis=axes[1])